# 第 25 天：Barra实战

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：Barra实战
> 必做：因子暴露
> 选做：收益归因
> 目标产出：风险归因图

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 估计每日风格因子收益和行业收益。
2. 计算组合对风格和行业的暴露。
3. 把组合收益拆成风格贡献、行业贡献和残差贡献。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

策略赚钱之后，第一句话不是“我真厉害”，而是“钱到底从哪里来的”。Barra 实战就是把收益拆成风格、行业和特质三张账单。

## 5. 今日核心实验


### 实验 1：先构建一个示例多因子组合

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
selected = ["value", "quality", "momentum_20", "low_vol", "liquidity"]
score = make_equal_weight_composite(factor_library, selected)
weights = make_market_neutral_weights(score, q=0.2)
strategy_ret = portfolio_return(weights, returns)

print(strategy_ret.describe().round(5))


### 实验 2：估计每日 Barra 因子收益

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
style_names = ["size", "value", "momentum_20", "low_vol", "liquidity"]
industry_exposure = pd.get_dummies(industries)
industry_exposure = industry_exposure.drop(columns=industry_exposure.columns[-1])

factor_return_rows = []
residual_rows = []

for dt in dates[80:-2]:
    style_exp = pd.DataFrame({name: factor_library[name].loc[dt] for name in style_names})
    X_dt = pd.concat([style_exp, industry_exposure], axis=1).astype(float)
    y_dt = returns.shift(-1).loc[dt].reindex(X_dt.index)
    valid = y_dt.notna() & X_dt.notna().all(axis=1)
    if valid.sum() < X_dt.shape[1] + 5:
        continue
    design = np.column_stack([np.ones(valid.sum()), X_dt.loc[valid].to_numpy()])
    coef = np.linalg.lstsq(design, y_dt.loc[valid].to_numpy(), rcond=None)[0]
    pred = design @ coef
    residual = y_dt.loc[valid] - pred
    factor_return_rows.append(pd.Series(coef[1:], index=X_dt.columns, name=dt))
    residual_rows.append(residual.rename(dt))

barra_factor_returns = pd.DataFrame(factor_return_rows)
print(barra_factor_returns[style_names].head().round(5))


### 实验 3：组合暴露：我的组合到底押了什么

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
portfolio_exposure_rows = []
for dt in barra_factor_returns.index:
    w = weights.loc[dt].reindex(assets).fillna(0)
    style_exp = pd.DataFrame({name: factor_library[name].loc[dt] for name in style_names})
    X_dt = pd.concat([style_exp, industry_exposure], axis=1).astype(float)
    portfolio_exposure_rows.append((X_dt.mul(w, axis=0)).sum().rename(dt))

portfolio_exposure = pd.DataFrame(portfolio_exposure_rows)
print(portfolio_exposure[style_names].describe().round(4))


### 实验 4：收益归因：风格、行业和残差

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
common_index = portfolio_exposure.index.intersection(barra_factor_returns.index)
factor_contribution = portfolio_exposure.loc[common_index] * barra_factor_returns.loc[common_index]
style_contribution = factor_contribution[style_names].sum(axis=1)
industry_cols = [c for c in factor_contribution.columns if c not in style_names]
industry_contribution = factor_contribution[industry_cols].sum(axis=1)
actual = strategy_ret.reindex(common_index)
residual_contribution = actual - style_contribution - industry_contribution

attribution = pd.DataFrame({
    "actual": actual,
    "style": style_contribution,
    "industry": industry_contribution,
    "residual": residual_contribution,
}).dropna()

print(attribution.describe().round(5))


### 实验 5：风险归因图：收益来自哪里

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
attribution.cumsum().plot(figsize=(10, 4), title="组合收益归因累计")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()
plt.close()

print("累计贡献：")
print(attribution.sum().round(4))


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：Barra实战
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：用当日收盘后才知道的权重解释当日收益。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：忽略组合权重归一化，导致暴露不可比。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：只看平均暴露，不看暴露随时间漂移。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：归因残差过大却不追查模型缺漏。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 26 天进入机器学习因子：先做特征工程和训练数据集。

## 13. 一句话收尾

Barra实战 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本、严格样本外检验和风险约束。
